# 03 — 指標可視化・モデル間比較・考察

各Notebookは独立実行できます。このNotebookも `run_experiment` でPopularityを評価し、保存済み結果を追加して比較できます。
NDCG・Recallの計算は共通評価器に任せ、ここでは集計値・要求ごとの差・実行時間を可視化します。
既定では合成データのみを使い、MLflowの記録・読込みは無効です。

| 入力 | 形式 |
| --- | --- |
| 今回の実行 | `BenchmarkReport` |
| ローカル保存 | `save_report` の `results.json` |
| MLflow成果物 | 集計済み `reports/report.json` |

比較は同じ `dataset_id`・`dataset_sha256`・`top_k` に限定します。
現行入力JSONのSHA-256は学習履歴・要求・正解を含む明示的分割全体を識別します。
異なる分割、ハッシュのない結果、異なるKは `compare_reports` が拒否します。
今後データ形式を変える場合は、分割・候補集合を識別するprovenanceも分析APIへ渡す必要があります。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

from music_recsys.analysis import compare_reports, request_rows
from music_recsys.evaluation.benchmark import run_experiment, save_report
from music_recsys.evaluation.tracking import TrackingConfig

ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "configs/toy.toml").is_file()
)

TRACK_EXPERIMENT = False
tracking = (
    TrackingConfig.from_settings({"enabled": True})
    if TRACK_EXPERIMENT
    else TrackingConfig(enabled=False)
)
report = run_experiment(ROOT / "configs/toy.toml", tracking=tracking)
RUN_DIR = save_report(report, ROOT / "experiments/toy/runs")
requests = request_rows(report)
display(
    {
        "dataset": report.metadata.dataset_id,
        "top_k": report.metadata.top_k,
        "requests": report.summary.n_requests,
        "evaluated": report.summary.n_evaluated,
        "excluded": report.summary.n_excluded,
        "failed": report.summary.n_failed,
        "short_outputs": report.summary.n_short,
    }
)
print("Saved report:", RUN_DIR / "results.json")

## 要求別の指標と集計対象

候補内に正解がない要求のNDCG・Recallは未定義（`None`）で、平均から除外します。
未定義を0に置き換えません。正解があって推薦が失敗した要求は0として含め、失敗数にも残します。
合成データの要求5は正解なしです。Cold userを含む動作確認で、Music4AllのWarm対象評価とは異なります。

In [ ]:
display(requests)
evaluated = [row for row in requests if row["ndcg"] is not None]
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
indices = [row["request_index"] for row in evaluated]
axes[0].bar(
    [i - 0.18 for i in indices],
    [row["ndcg"] for row in evaluated],
    width=0.36,
    color="#256b78",
    label="NDCG",
)
axes[0].bar(
    [i + 0.18 for i in indices],
    [row["recall"] for row in evaluated],
    width=0.36,
    color="#c77936",
    label="Recall",
)
axes[0].set(
    title=f"Evaluated requests, K={report.metadata.top_k}",
    xlabel="Request index",
    ylabel="Score",
    ylim=(0, 1.05),
    xticks=indices,
)
axes[0].legend()
axes[1].bar(
    ["Evaluated", "Excluded", "Failed"],
    [report.summary.n_evaluated, report.summary.n_excluded, report.summary.n_failed],
    color=["#256b78", "#c77936", "#8b507b"],
)
axes[1].set(title="Evaluation population and failures", ylabel="Requests")
fig.tight_layout()
plt.show()

## 任意：保存済みreport・MLflow成果物の追加

`ADDITIONAL_REPORTS` に表示名と `Path` を追加します。別モデルを実装・実行して同条件で保存した結果もここに追加できます。
現時点の実装はPopularityだけなので、既定の比較図は1モデルです。

MLflowを読む場合は `READ_MLFLOW=True`・`MLFLOW_RUN_ID` を設定し、サーバーを起動します。
`MlflowClient` でそのrunの集計成果物をダウンロードします。ユーザー別結果・生履歴をMLflowへ送信しません。
接続先は `TrackingConfig` の環境変数/設定を使います。詳細は `docs/mlflow.md` にあります。

In [ ]:
ADDITIONAL_REPORTS = {}  # 例: {"saved-popularity": Path("/absolute/path/results.json")}
READ_MLFLOW = False
MLFLOW_RUN_ID = None

reports = {"Popularity (current)": report, **ADDITIONAL_REPORTS}
if READ_MLFLOW:
    from mlflow import MlflowClient

    if MLFLOW_RUN_ID is None:
        raise ValueError("Set MLFLOW_RUN_ID before reading an MLflow report")
    client = MlflowClient(tracking_uri=TrackingConfig.from_settings().tracking_uri)
    mlflow_dir = RUN_DIR / "mlflow-download"
    mlflow_dir.mkdir(exist_ok=True)
    artifact_path = client.download_artifacts(MLFLOW_RUN_ID, "reports/report.json", str(mlflow_dir))
    reports["MLflow saved run"] = Path(artifact_path)

comparison = compare_reports(reports)
display(comparison)

## 共通条件での指標・実行時間

モデル名・seed・学習時間は比較表に残します。条件の不一致はグラフ作成前にエラーになります。
実行時間は実測値で、OSの負荷・計測の粒度に影響されます。短時間の合成実験から速度の優劣を結論付けません。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
labels = [row["label"] for row in comparison]
for ax, metric, title in zip(
    axes,
    ("ndcg", "recall", "mean_recommendation_seconds"),
    (
        f"NDCG@{report.metadata.top_k}",
        f"Recall@{report.metadata.top_k}",
        "Mean recommendation seconds",
    ),
):
    values = [row[metric] for row in comparison]
    defined = [(label, value) for label, value in zip(labels, values) if value is not None]
    ax.bar([label for label, _ in defined], [value for _, value in defined], color="#256b78")
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=15)
    if metric in ("ndcg", "recall"):
        ax.set_ylim(0, 1.05)
    if not defined:
        ax.text(0.5, 0.5, "Undefined: no evaluated requests", ha="center", transform=ax.transAxes)
fig.tight_layout()
plt.show()

## 考察・次の比較

- 合成データでのPopularityは、NDCG@2 = 0.561019、Recall@2 = 0.625。5要求のうち4要求を平均に含める。
- 人気曲を優先する結果・履歴除外・対象数は `02_recommendation.ipynb` と要求別表から確認できる。
- Music4Allでのモデル比較は未実施。時系列境界・最小履歴量・最終再学習の条件は `01_data.ipynb` の候補検証をもとに決める。
- 別モデルの結果が揃ったら同じデータ・分割・Kで比較する。LLM/Agentの費用や説明品質は未測定で、今回の図に推定値を足さない。